# 🛠️ Notebook 2: Facebook (core social graph) — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/facebook
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from itertools import count
from typing import Optional

class ReactionType(Enum):
    LIKE="👍"; LOVE="❤️"; HAHA="😂"; WOW="😮"; SAD="😢"; ANGRY="😡"

_pid = count(1); _cid = count(1); _uid = count(1)

@dataclass
class User:
    name: str
    id: int = field(default_factory=lambda: next(_uid))
    friends: set["User"] = field(default_factory=set)

    def __hash__(self): return self.id
    def __eq__(self, o): return isinstance(o, User) and self.id == o.id
    def __repr__(self): return f"User({self.name})"

    def add_friend(self, other: "User"):
        if other is self: return
        self.friends.add(other)
        other.friends.add(self)           # symmetric


In [ ]:
@dataclass
class Comment:
    author: User
    text: str
    ts: datetime = field(default_factory=datetime.utcnow)
    id: int = field(default_factory=lambda: next(_cid))

@dataclass
class Post:
    author: User
    content: str
    ts: datetime = field(default_factory=datetime.utcnow)
    id: int = field(default_factory=lambda: next(_pid))
    comments: list[Comment] = field(default_factory=list)
    # user.id -> ReactionType (each user has at most one reaction)
    reactions: dict[int, ReactionType] = field(default_factory=dict)

    def react(self, user: User, r: ReactionType):
        self.reactions[user.id] = r     # upsert

    def remove_reaction(self, user: User):
        self.reactions.pop(user.id, None)

    def comment(self, user: User, text: str):
        self.comments.append(Comment(user, text))

    def reaction_summary(self) -> dict[ReactionType, int]:
        out: dict[ReactionType, int] = {}
        for r in self.reactions.values():
            out[r] = out.get(r, 0) + 1
        return out


In [ ]:
class NewsFeed:
    def __init__(self, all_posts: list[Post]):
        self._all = all_posts

    def for_user(self, user: User, limit: int = 10) -> list[Post]:
        visible_authors = {user} | user.friends
        posts = [p for p in self._all if p.author in visible_authors]
        return sorted(posts, key=lambda p: p.ts, reverse=True)[:limit]


# demo
alice = User("Alice"); bob = User("Bob"); carol = User("Carol")
alice.add_friend(bob)           # symmetric

all_posts: list[Post] = []
p1 = Post(alice, "hello world");                all_posts.append(p1)
p2 = Post(bob,   "hi alice!");                  all_posts.append(p2)
p3 = Post(carol, "carol is a stranger here");   all_posts.append(p3)

p1.react(bob, ReactionType.LOVE)
p1.comment(bob, "❤️")
p2.react(alice, ReactionType.LIKE)

feed = NewsFeed(all_posts)
print("Alice's feed (should NOT include Carol):")
for p in feed.for_user(alice):
    print(" ", p.author, ":", p.content, "→ reactions:", p.reaction_summary())

print("\nCarol's feed (only her own posts — no friends):")
for p in feed.for_user(carol):
    print(" ", p.author, ":", p.content)


### Try it
- Add **Groups**: a post can be authored to a group; group members can see it.
- Add **Pages** (one-way follow) — break the symmetry.
- Replace `NewsFeed` with a **fanout-on-write** version: push new posts into each friend's inbox at post time.